In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import norm, skewnorm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings, os, time, json

warnings.filterwarnings("ignore")

# ── Configuration ──
MODEL_NAME = "Pinball"
DATASET_NAME = "Synthetic_Normal"
RESULTS_DIR = "results"
PLOTS_DIR = "plots"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

SPLIT_SEED = 58
SEEDS = [42, 58, 123, 256, 789]
N_RUNS = len(SEEDS)
DEVICE = torch.device("cpu")  # CPU faster for small MLPs

QUANTILES = [
    0.01, 0.025, 0.03, 0.05, 0.09, 0.10, 0.20, 0.30, 0.40, 0.50,
    0.60, 0.70, 0.80, 0.90, 0.91, 0.93, 0.95, 0.97, 0.975, 0.99,
]
OUTLIER_TYPES = ["gaussian", "multiply", "uniform", "skewed"]
CONTAMINATION_LEVELS = [0.02, 0.05, 0.10]
EP = 100
H = 100
LR = 0.01
BS = 32

# ── Plot Style ──
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "font.size": 11,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "figure.dpi": 150,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
})

C_PRED = "#2563EB"     # blue for predictions
C_TRUE = "#DC2626"     # red for true quantiles
C_CLEAN = "#3B82F6"    # light blue for clean data
C_CONTAM = "#F59E0B"   # amber for contaminated data
C_MODEL = "#7C3AED"    # purple accent
C_BAR = "#0EA5E9"      # sky blue for bar/line charts

print(f"Config: {MODEL_NAME} | {DATASET_NAME} | {N_RUNS} seeds | Device: {DEVICE}")


Config: Pinball | Synthetic_Normal | 5 seeds | Device: cpu


In [2]:
# ── Data Generation ──
np.random.seed(SPLIT_SEED)
N_SAMPLES = 1000
X_raw = np.random.uniform(-2 * np.pi, 2 * np.pi, N_SAMPLES)

def true_mean(x):
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(np.abs(x) < 1e-10, 1.0, np.sin(x) / x)

def true_quantile(x, tau):
    return true_mean(x) + norm.ppf(tau)

Y_clean = true_mean(X_raw) + np.random.normal(0, 1, N_SAMPLES)
X_col = X_raw.reshape(-1, 1)
DIM = 1

# Train / Val / Test split: 64% / 16% / 20%
Xtv, X_test, ytv, y_test = train_test_split(
    X_col, Y_clean, test_size=0.20, random_state=SPLIT_SEED
)
X_train, X_val, y_train_clean, y_val = train_test_split(
    Xtv, ytv, test_size=0.20, random_state=SPLIT_SEED
)

print(f"Split: Train={X_train.shape[0]} Val={X_val.shape[0]} Test={X_test.shape[0]}")


Split: Train=640 Val=160 Test=200


In [3]:
# ── Outlier Injection ──
def inject_outliers(y, frac, method, seed=None):
    rng = np.random.RandomState(seed)
    y = y.copy()
    n = len(y)
    n_out = max(1, int(frac * n))
    idx = rng.choice(n, n_out, replace=False)
    mu, sig = y.mean(), y.std()
    if method == "gaussian":
        y[idx] = rng.normal(mu, np.sqrt(5) * sig, n_out)
    elif method == "multiply":
        y[idx] = y[idx] * rng.choice([-1, 1], n_out) * 10
    elif method == "uniform":
        y[idx] = rng.uniform(y.min() - 3 * sig, y.max() + 3 * sig, n_out)
    elif method == "skewed":
        y[idx] = skewnorm.rvs(a=10, loc=mu, scale=3 * sig, size=n_out, random_state=rng)
    return y

# Build all contaminated training sets
cont_sets = {}
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        cont_sets[(ot, cl)] = inject_outliers(y_train_clean, cl, ot, seed=SPLIT_SEED)

print(f"{len(cont_sets)} contaminated training sets created")


12 contaminated training sets created


In [4]:
# ── Scatter Plots: Clean vs Contaminated ──

# Clean-only scatter
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train.ravel(), y_train_clean, s=10, alpha=0.5, color=C_CLEAN, label="Clean Data")
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title(f"{DATASET_NAME} — Clean Training Data")
ax.legend(frameon=False)
plt.savefig(f"{PLOTS_DIR}/scatter_clean.png")
plt.close()

# Clean vs Contaminated side-by-side for each type × level
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].scatter(X_train.ravel(), y_train_clean, s=10, alpha=0.5, color=C_CLEAN)
        axes[0].set_title("Clean Data"); axes[0].set_xlabel("X"); axes[0].set_ylabel("Y")

        axes[1].scatter(X_train.ravel(), cont_sets[(ot, cl)], s=10, alpha=0.5, color=C_CONTAM)
        axes[1].set_title(f"{ot.capitalize()} {int(cl*100)}% Contamination")
        axes[1].set_xlabel("X"); axes[1].set_ylabel("Y")

        for ax in axes:
            ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

        plt.suptitle(f"{DATASET_NAME} — {ot.capitalize()} {int(cl*100)}%", fontsize=13, y=1.01)
        plt.tight_layout()
        plt.savefig(f"{PLOTS_DIR}/scatter_{ot}_{int(cl*100)}pct.png")
        plt.close()

print(f"Saved {1 + len(cont_sets)} scatter plots")


Saved 13 scatter plots


In [5]:
# ── Model & Loss ──
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, H), nn.ReLU(), nn.Linear(H, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def pinball_loss(pred, target, tau):
    e = target - pred
    return torch.mean(torch.max(tau * e, (tau - 1) * e))

def set_seed(s):
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def train_model(X, y, loss_fn, epochs=EP):
    model = MLP(DIM).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=LR)
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    ds = torch.utils.data.TensorDataset(Xt, yt)
    dl = torch.utils.data.DataLoader(ds, batch_size=BS, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()
    model.eval()
    return model

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()

print("Model and loss functions defined")


Model and loss functions defined


In [6]:
# ── Evaluation ──
def compute_shift_rmse(preds_clean, preds_contam, quantiles):
    """RMSE between clean and contaminated predictions, per quantile."""
    results = {}
    for tau in quantiles:
        diff = preds_clean[tau] - preds_contam[tau]
        results[tau] = np.sqrt(np.mean(diff ** 2))
    return results

def compute_ece(y_test, preds, quantiles):
    """Estimated Calibration Error per quantile."""
    results = {}
    for tau in quantiles:
        coverage = np.mean(y_test <= preds[tau])
        results[tau] = abs(coverage - tau)
    return results

def compute_true_quantile_rmse(preds, x_test, quantiles):
    """RMSE vs known true conditional quantiles (synthetic only)."""
    results = {}
    for tau in quantiles:
        tq = true_quantile(x_test.ravel(), tau)
        results[tau] = np.sqrt(mean_squared_error(tq, preds[tau]))
    return results

print("Evaluation functions defined")


Evaluation functions defined


In [7]:
# ── Main Experiment ──
# Storage: all_results[seed][(condition)] = {tau: predictions}
# condition = "clean" or ("gaussian", 0.02) etc.
all_preds = {}          # seed -> condition -> tau -> predictions array
all_shift_rmse = {}     # seed -> condition -> tau -> value
all_ece_clean = {}      # seed -> tau -> value
all_ece_contam = {}     # seed -> condition -> tau -> value
all_true_q_rmse = {}    # seed -> tau -> value (clean only)

conditions = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]
total = N_RUNS * len(conditions) * len(QUANTILES)
print(f"Total models to train: {total}")

t0 = time.time()
for si, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"Seed {seed} ({si+1}/{N_RUNS})")
    print(f"{'='*60}")

    all_preds[seed] = {}
    all_shift_rmse[seed] = {}
    all_ece_contam[seed] = {}

    for ci, cond in enumerate(conditions):
        cond_label = "clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
        y_tr = y_train_clean if cond == "clean" else cont_sets[cond]

        preds = {}
        for tau in QUANTILES:
            set_seed(seed)
            model = train_model(X_train, y_tr, lambda p, y, t=tau: pinball_loss(p, y, t))
            preds[tau] = predict(model, X_test)

        all_preds[seed][cond] = preds

        if cond == "clean":
            all_ece_clean[seed] = compute_ece(y_test, preds, QUANTILES)
            all_true_q_rmse[seed] = compute_true_quantile_rmse(preds, X_test, QUANTILES)
        else:
            all_shift_rmse[seed][cond] = compute_shift_rmse(
                all_preds[seed]["clean"], preds, QUANTILES
            )
            all_ece_contam[seed][cond] = compute_ece(y_test, preds, QUANTILES)

        elapsed = time.time() - t0
        done = si * len(conditions) * len(QUANTILES) + (ci + 1) * len(QUANTILES)
        pct = done / total * 100
        print(f"  [{pct:5.1f}%] {cond_label:25s} | {elapsed/60:.1f}min elapsed")

total_time = time.time() - t0
print(f"\nDone: {total_time/60:.1f} minutes total")


Total models to train: 1300

Seed 42 (1/5)


  [  1.5%] clean                     | 0.4min elapsed


  [  3.1%] gaussian_2pct             | 0.9min elapsed


  [  4.6%] gaussian_5pct             | 1.3min elapsed


  [  6.2%] gaussian_10pct            | 1.8min elapsed


...

  [ 93.8%] uniform_5pct              | 26.1min elapsed


  [ 95.4%] uniform_10pct             | 26.6min elapsed


  [ 96.9%] skewed_2pct               | 27.0min elapsed


  [ 98.5%] skewed_5pct               | 27.4min elapsed


  [100.0%] skewed_10pct              | 27.8min elapsed

Done: 27.8 minutes total


In [8]:
# ── Save Results to Excel ──
from openpyxl import Workbook

wb = Workbook()
ws_summary = wb.active
ws_summary.title = "Summary"

contam_conditions = [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

# ── Per-seed sheets: Shift-RMSE ──
for seed in SEEDS:
    ws = wb.create_sheet(title=f"ShiftRMSE_seed_{seed}")
    header = ["Condition"] + [str(t) for t in QUANTILES] + ["Avg"]
    ws.append(header)
    for cond in contam_conditions:
        label = f"{cond[0]}_{int(cond[1]*100)}pct"
        vals = all_shift_rmse[seed][cond]
        row = [label] + [round(vals[t], 6) for t in QUANTILES]
        row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
        ws.append(row)

# ── Per-seed sheets: ECE (clean) ──
for seed in SEEDS:
    ws = wb.create_sheet(title=f"ECE_clean_seed_{seed}")
    header = ["Metric"] + [str(t) for t in QUANTILES] + ["Avg"]
    ws.append(header)
    vals = all_ece_clean[seed]
    row = ["ECE_clean"] + [round(vals[t], 6) for t in QUANTILES]
    row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
    ws.append(row)

# ── Per-seed sheets: ECE (contaminated) ──
for seed in SEEDS:
    ws = wb.create_sheet(title=f"ECE_contam_seed_{seed}")
    header = ["Condition"] + [str(t) for t in QUANTILES] + ["Avg"]
    ws.append(header)
    for cond in contam_conditions:
        label = f"{cond[0]}_{int(cond[1]*100)}pct"
        vals = all_ece_contam[seed][cond]
        row = [label] + [round(vals[t], 6) for t in QUANTILES]
        row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
        ws.append(row)

# ── Per-seed sheets: True Quantile RMSE ──
for seed in SEEDS:
    ws = wb.create_sheet(title=f"TrueQ_RMSE_seed_{seed}")
    header = ["Metric"] + [str(t) for t in QUANTILES] + ["Avg"]
    ws.append(header)
    vals = all_true_q_rmse[seed]
    row = ["TrueQ_RMSE"] + [round(vals[t], 6) for t in QUANTILES]
    row.append(round(np.mean([vals[t] for t in QUANTILES]), 6))
    ws.append(row)

# ── Summary sheet: means ± std ──
ws_summary.append(["=== Shift-RMSE (mean ± std across seeds) ==="])
header = ["Condition"] + [str(t) for t in QUANTILES] + ["Avg"]
ws_summary.append(header)
for cond in contam_conditions:
    label = f"{cond[0]}_{int(cond[1]*100)}pct"
    means = []
    for t in QUANTILES:
        vals = [all_shift_rmse[s][cond][t] for s in SEEDS]
        means.append(np.mean(vals))
    row = [label] + [round(m, 6) for m in means] + [round(np.mean(means), 6)]
    ws_summary.append(row)

ws_summary.append([])
ws_summary.append(["=== ECE Clean (mean ± std across seeds) ==="])
header = ["Metric"] + [str(t) for t in QUANTILES] + ["Avg"]
ws_summary.append(header)
means_ece = []
for t in QUANTILES:
    vals = [all_ece_clean[s][t] for s in SEEDS]
    means_ece.append(np.mean(vals))
ws_summary.append(["ECE_clean"] + [round(m, 6) for m in means_ece] + [round(np.mean(means_ece), 6)])

ws_summary.append([])
ws_summary.append(["=== True Quantile RMSE (mean across seeds) ==="])
header = ["Metric"] + [str(t) for t in QUANTILES] + ["Avg"]
ws_summary.append(header)
means_tq = []
for t in QUANTILES:
    vals = [all_true_q_rmse[s][t] for s in SEEDS]
    means_tq.append(np.mean(vals))
ws_summary.append(["TrueQ_RMSE"] + [round(m, 6) for m in means_tq] + [round(np.mean(means_tq), 6)])

# ── Alpha sheet (for Pinball: no alpha, just record N/A) ──
ws_alpha = wb.create_sheet(title="Alpha")
ws_alpha.append(["Method", "Alpha"])
ws_alpha.append([MODEL_NAME, "N/A (no trimming)"])

xlsx_path = f"{RESULTS_DIR}/{DATASET_NAME}_{MODEL_NAME}_results.xlsx"
wb.save(xlsx_path)
print(f"Saved: {xlsx_path}")


Saved: results/Synthetic_Normal_Pinball_results.xlsx


In [9]:
# ── Quantile Curve Plots (tau=0.025 and tau=0.975) ──
sort_idx = np.argsort(X_test.ravel())
X_sorted = X_test.ravel()[sort_idx]
tq_lo = true_quantile(X_sorted, 0.025)
tq_hi = true_quantile(X_sorted, 0.975)

def plot_quantile_curves(preds, y_train_plot, title_suffix, fname, show_data=True):
    fig, ax = plt.subplots(figsize=(10, 6))
    if show_data:
        ax.scatter(X_train.ravel(), y_train_plot, s=8, alpha=0.3, color=C_CONTAM,
                   label="Training Data", zorder=1)

    pred_lo = preds[0.025][sort_idx]
    pred_hi = preds[0.975][sort_idx]

    ax.plot(X_sorted, tq_lo, "-", color=C_TRUE, lw=1.5, label="True Q(0.025)", zorder=2)
    ax.plot(X_sorted, pred_lo, "--", color=C_PRED, lw=1.5, label="Pred Q(0.025)", zorder=3)
    ax.plot(X_sorted, tq_hi, "-", color=C_TRUE, lw=1.5, label="True Q(0.975)", zorder=2)
    ax.plot(X_sorted, pred_hi, "--", color=C_PRED, lw=1.5, label="Pred Q(0.975)", zorder=3)

    ax.set_xlabel("X"); ax.set_ylabel("Y")
    ax.set_title(f"{MODEL_NAME} — {title_suffix}", fontsize=12)
    ax.legend(frameon=False, fontsize=9, loc="upper right")
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/{fname}")
    plt.close()

# Per-seed and averaged plots
conditions_plot = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for cond in conditions_plot:
    if cond == "clean":
        cond_label = "Clean"
        y_train_plot = y_train_clean
    else:
        cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
        y_train_plot = cont_sets[cond]

    # Per-seed plots
    for seed in SEEDS:
        preds = all_preds[seed][cond]
        fname_tag = f"clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
        plot_quantile_curves(
            preds, y_train_plot,
            f"{cond_label} (seed={seed})",
            f"qcurve_{fname_tag}_seed{seed}.png"
        )

    # Averaged predictions across seeds
    avg_preds = {}
    for tau in QUANTILES:
        avg_preds[tau] = np.mean([all_preds[s][cond][tau] for s in SEEDS], axis=0)

    fname_tag = f"clean" if cond == "clean" else f"{cond[0]}_{int(cond[1]*100)}pct"
    plot_quantile_curves(
        avg_preds, y_train_plot,
        f"{cond_label} (avg {N_RUNS} seeds)",
        f"qcurve_{fname_tag}_avg.png"
    )

print(f"Saved quantile curve plots: {len(conditions_plot)} conditions × ({N_RUNS} seeds + 1 avg)")


Saved quantile curve plots: 13 conditions × (5 seeds + 1 avg)


In [10]:
# ── Shift-RMSE per Quantile Plots ──
contam_conditions = [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]

for cond in contam_conditions:
    cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
    fname_tag = f"{cond[0]}_{int(cond[1]*100)}pct"

    # Gather per-seed values
    per_seed_vals = np.array([
        [all_shift_rmse[s][cond][t] for t in QUANTILES] for s in SEEDS
    ])  # shape: (N_RUNS, n_quantiles)
    means = per_seed_vals.mean(axis=0)
    stds = per_seed_vals.std(axis=0)

    # Averaged with error bars
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color=C_BAR, lw=2,
                markersize=5, capsize=3, capthick=1, label=f"{MODEL_NAME} (mean ± std)")
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("Shift-RMSE")
    ax.set_title(f"{DATASET_NAME} — Shift-RMSE per Quantile: {MODEL_NAME} ({cond_label})")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/shift_rmse_{fname_tag}_avg.png")
    plt.close()

    # Per-seed plots
    for si, seed in enumerate(SEEDS):
        fig, ax = plt.subplots(figsize=(14, 5))
        vals = [all_shift_rmse[seed][cond][t] for t in QUANTILES]
        ax.plot(x_ticks, vals, "o-", color=C_BAR, lw=2, markersize=5)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Quantile τ"); ax.set_ylabel("Shift-RMSE")
        ax.set_title(f"{DATASET_NAME} — Shift-RMSE: {MODEL_NAME} ({cond_label}, seed={seed})")
        plt.tight_layout()
        plt.savefig(f"{PLOTS_DIR}/shift_rmse_{fname_tag}_seed{seed}.png")
        plt.close()

print(f"Saved Shift-RMSE plots: {len(contam_conditions)} conditions × ({N_RUNS} + 1) = {len(contam_conditions)*(N_RUNS+1)}")


Saved Shift-RMSE plots: 12 conditions × (5 + 1) = 72


In [11]:
# ── RMSE vs True Quantiles (Clean — Adaptiveness) ──
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]

per_seed_vals = np.array([
    [all_true_q_rmse[s][t] for t in QUANTILES] for s in SEEDS
])
means = per_seed_vals.mean(axis=0)
stds = per_seed_vals.std(axis=0)

# Averaged with error bars
fig, ax = plt.subplots(figsize=(14, 5))
ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color=C_MODEL, lw=2,
            markersize=5, capsize=3, capthick=1, label=f"{MODEL_NAME} (mean ± std)")
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Quantile τ"); ax.set_ylabel("RMSE vs True Quantile")
ax.set_title(f"{DATASET_NAME} — RMSE vs True Quantiles (Clean): {MODEL_NAME}")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/true_q_rmse_clean_avg.png")
plt.close()

# Per-seed
for seed in SEEDS:
    fig, ax = plt.subplots(figsize=(14, 5))
    vals = [all_true_q_rmse[seed][t] for t in QUANTILES]
    ax.plot(x_ticks, vals, "o-", color=C_MODEL, lw=2, markersize=5)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("RMSE vs True Quantile")
    ax.set_title(f"{DATASET_NAME} — RMSE vs True Quantiles (Clean): {MODEL_NAME} (seed={seed})")
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/true_q_rmse_clean_seed{seed}.png")
    plt.close()

print(f"Saved True Quantile RMSE plots: {N_RUNS} + 1 = {N_RUNS + 1}")


Saved True Quantile RMSE plots: 5 + 1 = 6


In [12]:
# ── ECE per Quantile ──
x_ticks = list(range(len(QUANTILES)))
x_labels = [str(q) for q in QUANTILES]

all_ece_conditions = ["clean"] + [(ot, cl) for ot in OUTLIER_TYPES for cl in CONTAMINATION_LEVELS]

for cond in all_ece_conditions:
    if cond == "clean":
        cond_label = "Clean"
        fname_tag = "clean"
        per_seed_vals = np.array([
            [all_ece_clean[s][t] for t in QUANTILES] for s in SEEDS
        ])
    else:
        cond_label = f"{cond[0].capitalize()} {int(cond[1]*100)}%"
        fname_tag = f"{cond[0]}_{int(cond[1]*100)}pct"
        per_seed_vals = np.array([
            [all_ece_contam[s][cond][t] for t in QUANTILES] for s in SEEDS
        ])

    means = per_seed_vals.mean(axis=0)
    stds = per_seed_vals.std(axis=0)

    # Averaged
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.errorbar(x_ticks, means, yerr=stds, fmt="o-", color="#10B981", lw=2,
                markersize=5, capsize=3, capthick=1, label=f"{MODEL_NAME} (mean ± std)")
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Quantile τ"); ax.set_ylabel("ECE")
    ax.set_title(f"{DATASET_NAME} — ECE per Quantile: {MODEL_NAME} ({cond_label})")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/ece_{fname_tag}_avg.png")
    plt.close()

    # Per-seed
    for seed in SEEDS:
        fig, ax = plt.subplots(figsize=(14, 5))
        if cond == "clean":
            vals = [all_ece_clean[seed][t] for t in QUANTILES]
        else:
            vals = [all_ece_contam[seed][cond][t] for t in QUANTILES]
        ax.plot(x_ticks, vals, "o-", color="#10B981", lw=2, markersize=5)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
        ax.set_xlabel("Quantile τ"); ax.set_ylabel("ECE")
        ax.set_title(f"{DATASET_NAME} — ECE: {MODEL_NAME} ({cond_label}, seed={seed})")
        plt.tight_layout()
        plt.savefig(f"{PLOTS_DIR}/ece_{fname_tag}_seed{seed}.png")
        plt.close()

print(f"Saved ECE plots: {len(all_ece_conditions)} conditions × ({N_RUNS} + 1)")


Saved ECE plots: 13 conditions × (5 + 1)


In [13]:
# ── Final Summary ──
print(f"{'='*60}")
print(f"{MODEL_NAME} on {DATASET_NAME} — Experiment Complete")
print(f"{'='*60}")
print(f"Seeds: {SEEDS}")
print(f"Quantiles: {len(QUANTILES)}")
print(f"Contamination settings: {len(cont_sets)}")
print(f"Total models trained: {N_RUNS * (1 + len(cont_sets)) * len(QUANTILES)}")
print(f"Results saved to: {RESULTS_DIR}/")
print(f"Plots saved to: {PLOTS_DIR}/")

# Quick summary table
print(f"\n--- Avg True Quantile RMSE (clean, across seeds) ---")
for tau in [0.025, 0.05, 0.10, 0.50, 0.90, 0.95, 0.975]:
    vals = [all_true_q_rmse[s][tau] for s in SEEDS]
    print(f"  τ={tau:.3f}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print(f"\n--- Avg ECE (clean, across seeds) ---")
avg_ece = np.mean([np.mean([all_ece_clean[s][t] for t in QUANTILES]) for s in SEEDS])
print(f"  Overall: {avg_ece:.4f}")

print(f"\n--- Avg Shift-RMSE (multiply 10%, across seeds) ---")
for tau in [0.025, 0.05, 0.10, 0.50, 0.90, 0.95, 0.975]:
    vals = [all_shift_rmse[s][("multiply", 0.10)][tau] for s in SEEDS]
    print(f"  τ={tau:.3f}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")


Pinball on Synthetic_Normal — Experiment Complete
Seeds: [42, 58, 123, 256, 789]
Quantiles: 20
Contamination settings: 12
Total models trained: 1300
Results saved to: results/
Plots saved to: plots/

--- Avg True Quantile RMSE (clean, across seeds) ---
  τ=0.025: 0.2595 ± 0.0642
  τ=0.050: 0.2444 ± 0.0437
  τ=0.100: 0.1741 ± 0.0156
  τ=0.500: 0.1940 ± 0.0627
  τ=0.900: 0.2111 ± 0.0493
  τ=0.950: 0.3141 ± 0.0872
  τ=0.975: 0.4982 ± 0.2166

--- Avg ECE (clean, across seeds) ---
  Overall: 0.0228

--- Avg Shift-RMSE (multiply 10%, across seeds) ---
  τ=0.025: 6.3195 ± 0.6062
  τ=0.050: 0.4953 ± 0.1463
  τ=0.100: 0.1685 ± 0.0200
  τ=0.500: 0.1306 ± 0.0487
  τ=0.900: 0.6490 ± 0.0845
  τ=0.950: 3.0751 ± 0.4431
  τ=0.975: 8.7294 ± 0.4602
